In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import os
import time
import tkinter as tk
from tkinter import scrolledtext, ttk, messagebox
import threading
import collections # Para collections.deque

# --- Constantes e Globais para Headers ---
BASE_URL_CAMBRIDGE = "https://dictionary.cambridge.org"
REQUEST_BASE_URL_DICTIONARY = "https://dictionary.cambridge.org/dictionary/english/"
DATA_FILE = "cambridge_dictionary_data.json"

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:126.0) Gecko/20100101 Firefox/126.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:126.0) Gecko/20100101 Firefox/126.0",
]
current_user_agent_index = 0

REQUEST_HEADERS = {
    "User-Agent": USER_AGENTS[0],
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "Accept-Language": "en-US,en;q=0.9,pt-BR;q=0.8,pt;q=0.7", # pt-BR adicionado para preferência
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
}

MAX_CONSECUTIVE_FETCH_ERRORS_BEFORE_UA_SWITCH = 3

shared_fetch_error_counter = [0] # [consecutive_fetch_errors]

REQUEST_DELAY_SECONDS = 5

# --- Funções Auxiliares de Parsing (sem alteração) ---
def safe_get_text(element, default=""):
    """Extrai o texto de um elemento BeautifulSoup de forma segura."""
    return element.get_text() if element else default

def safe_get_attr(element, attr, default=""):
    """Extrai um atributo de um elemento BeautifulSoup de forma segura."""
    return element.get(attr, default) if element else default

# --- Funções de Persistência de Dados (sem alteração significativa) ---
def load_existing_data(filepath, gui_app_instance=None):
    """Carrega dados de um arquivo JSON, se existir."""
    if os.path.exists(filepath):
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                return json.load(f)
        except json.JSONDecodeError:
            if gui_app_instance: gui_app_instance.log_message(f"Aviso: Arquivo '{filepath}' corrompido. Iniciando com dados vazios.")
            return {}
        except Exception as e:
            if gui_app_instance: gui_app_instance.log_message(f"Aviso: Não foi possível ler '{filepath}'. Erro: {e}. Iniciando com dados vazios.")
            return {}
    return {}

def save_data(data, filepath, gui_app_instance=None):
    try:
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
    except Exception as e:
        if gui_app_instance: gui_app_instance.log_message(f"Erro Crítico: Não foi possível salvar dados em '{filepath}'. Erro: {e}")

# --- Função de Requisição HTTP (Modificada) ---
def fetch_word_html(word_to_search, gui_app):
    global current_user_agent_index, REQUEST_HEADERS, shared_fetch_error_counter

    # Atualiza o header com o UA corrente (caso tenha sido trocado)
    REQUEST_HEADERS["User-Agent"] = USER_AGENTS[current_user_agent_index]
    
    url = f"{REQUEST_BASE_URL_DICTIONARY}{word_to_search.lower()}"
    gui_app.log_message(f"Tentando acessar: {url} (UA: ...{USER_AGENTS[current_user_agent_index][-40:]})")

    try:
        response = requests.get(url, headers=REQUEST_HEADERS, timeout=15)
        response.raise_for_status()
        gui_app.log_message(f"Status: {response.status_code} para '{word_to_search}'. Sucesso.")
        shared_fetch_error_counter[0] = 0 # Resetar contador de erros em sucesso
        return response.text
    except (requests.exceptions.ConnectionError, requests.exceptions.Timeout, requests.exceptions.HTTPError) as e:
        gui_app.log_message(f"Erro de fetch para '{word_to_search}': {type(e).__name__} - {e}")
        shared_fetch_error_counter[0] += 1
        if shared_fetch_error_counter[0] >= MAX_CONSECUTIVE_FETCH_ERRORS_BEFORE_UA_SWITCH:
            old_ua_index = current_user_agent_index
            current_user_agent_index = (current_user_agent_index + 1) % len(USER_AGENTS)
            REQUEST_HEADERS["User-Agent"] = USER_AGENTS[current_user_agent_index] # Atualiza o header global para a próxima chamada
            gui_app.log_message(f"Muitos erros de fetch! Trocando User-Agent de ...{USER_AGENTS[old_ua_index][-40:]} para ...{USER_AGENTS[current_user_agent_index][-40:]}")
            shared_fetch_error_counter[0] = 0
    except requests.exceptions.RequestException as e:
        gui_app.log_message(f"Erro na requisição para '{word_to_search}': {type(e).__name__} - {e}")
    except Exception as e:
        gui_app.log_message(f"Erro inesperado no fetch para '{word_to_search}': {type(e).__name__} - {e}")
    return None

# --- Função Principal de Parsing (sem alteração, apenas chamada) ---
def parse_cambridge_entry(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    entry_data = {
        "word": "", "part_of_speech": "", "grammar": "",
        "pronunciations": {"uk": {}, "us": {}},
        "senses": [],
        "smart_vocabulary": {"topic": {}, "related_words": []}
    }
    entry_body = soup.find('div', class_='pr entry-body__el')
    if not entry_body: return entry_data # Retorna vazio se o corpo principal não for encontrado

    pos_header = entry_body.find('div', class_='pos-header')
    if pos_header:
        headword_span = pos_header.find('span', class_='hw dhw')
        entry_data['word'] = safe_get_text(headword_span)
        pos_span = pos_header.find('span', class_='pos dpos')
        entry_data['part_of_speech'] = safe_get_text(pos_span)
        gram_span = pos_header.find('span', class_='gram dgram')
        entry_data['grammar'] = safe_get_text(gram_span)

        uk_pron_span = pos_header.find('span', class_='uk dpron-i')
        if uk_pron_span:
            uk_audio_src = safe_get_attr(uk_pron_span.find('source', type='audio/mpeg'), 'src')

            # Tentativa de encontrar o span do IPA
            ipa_element = uk_pron_span.find('span', class_='ipa dipa')
            
            # Log para depuração:
            # print(f"Elemento IPA com 'ipa dipa': {ipa_element}")

            if not ipa_element:
                # Se não encontrou com 'ipa dipa', tente apenas com 'ipa'
                # Isso é mais robusto se a classe 'dipa' nem sempre estiver presente ou variar
                ipa_element = uk_pron_span.find('span', class_='ipa')
                # Log para depuração:
                # print(f"Elemento IPA apenas com 'ipa': {ipa_element}")
            
            ipa_text = safe_get_text(ipa_element)
            # Se o ipa_text ainda estiver vindo com as barras, ex: "/ˈstɔː.ri/", você pode limpá-las:
            if ipa_text.startswith('/') and ipa_text.endswith('/'):
               ipa_text = ipa_text.strip('/')
            # No entanto, com o seletor correto para o span interno, isso não deve ser necessário.

            entry_data['pronunciations']['uk'] = {
                'ipa': ipa_text,
                'audio': BASE_URL_CAMBRIDGE + uk_audio_src if uk_audio_src else ""
            }

        us_pron_span = pos_header.find('span', class_='us dpron-i')
        if us_pron_span:
            us_audio_src = safe_get_attr(us_pron_span.find('source', type='audio/mpeg'), 'src')
            
            ipa_element_us = us_pron_span.find('span', class_='ipa dipa')
            if not ipa_element_us:
                ipa_element_us = us_pron_span.find('span', class_='ipa')
            
            ipa_text_us = safe_get_text(ipa_element_us)

            if ipa_text_us.startswith('/') and ipa_text_us.endswith('/'):
               ipa_text_us = ipa_text_us.strip('/')

            entry_data['pronunciations']['us'] = {
                'ipa': ipa_text_us,
                'audio': BASE_URL_CAMBRIDGE + us_audio_src if us_audio_src else ""
            }
    else: # Fallback para a palavra se o cabeçalho não for encontrado
        headword_fallback = entry_body.find('span', class_='hw dhw')
        if headword_fallback: entry_data['word'] = safe_get_text(headword_fallback)

    pos_body = entry_body.find('div', class_='pos-body')
    if pos_body:
        for def_block in pos_body.find_all('div', class_='def-block ddef_block'):
            current_sense = {}
            ddef_h = def_block.find('div', class_='ddef_h')
            if ddef_h:
                epp_xref = ddef_h.find('span', class_='epp-xref')
                current_sense['cefr_level'] = safe_get_text(epp_xref)
            current_sense['definition'] = safe_get_text(def_block.find('div', class_='def ddef_d db'))
            
            examples, more_examples, see_also_terms = [], [], []
            def_body_ddef_b = def_block.find('div', class_='def-body ddef_b')
            if def_body_ddef_b:
                for ex_div in def_body_ddef_b.find_all('div', class_='examp dexamp', recursive=False):
                    examples.append(safe_get_text(ex_div.find('span', class_='eg deg')))
                
                see_xref_div = def_body_ddef_b.find('div', class_='xref see hax dxref-w')
                if see_xref_div:
                    for item_div in see_xref_div.find_all('div', class_='item lc'):
                        term_url = safe_get_attr(item_div.find('a'), 'href')
                        see_also_terms.append({
                            "term": safe_get_text(item_div.find('span', class_='x-h dx-h')),
                            "url": BASE_URL_CAMBRIDGE + term_url if term_url and not term_url.startswith('http') else term_url
                        })
            current_sense['examples'] = [ex for ex in examples if ex] # Remove exemplos vazios
            
            daccord_more_examples = def_block.find('div', class_='daccord')
            if daccord_more_examples and safe_get_text(daccord_more_examples.find('span', class_='showmore')) == "More examples":
                for li_tag in daccord_more_examples.find_all('li', class_='eg dexamp hax'):
                    more_examples.append(safe_get_text(li_tag))
            current_sense['more_examples'] = [ex for ex in more_examples if ex] # Remove exemplos vazios
            current_sense['see_also'] = see_also_terms
            
            if current_sense.get('definition') or current_sense.get('examples'):
                entry_data['senses'].append(current_sense)

    smart_vocab_div = entry_body.find('div', class_='smartt daccord')
    if smart_vocab_div:
        topic_anchor = smart_vocab_div.find('div', class_='daccord_lt').find('a') if smart_vocab_div.find('div', class_='daccord_lt') else None
        if topic_anchor:
            entry_data['smart_vocabulary']['topic'] = {
                "name": safe_get_text(topic_anchor), "url": safe_get_attr(topic_anchor, 'href')
            }
        related_words_list = smart_vocab_div.find('ul', class_='hul-u')
        if related_words_list:
            for li_tag in related_words_list.find_all('li', class_='lc'):
                word_link_tag = li_tag.find('a')
                if word_link_tag:
                    word_text = ""
                    base_span = word_link_tag.find('span', class_='base')
                    if base_span:
                        text_parts = [s.get_text() for s in base_span.find_all(True, recursive=False) if s.get_text()]
                        word_text = ' '.join(text_parts) if text_parts else safe_get_text(base_span)
                    else:
                        results_span = word_link_tag.find('span', class_='results')
                        word_text = safe_get_text(results_span) if results_span else safe_get_text(word_link_tag)
                    
                    if word_text:
                        entry_data['smart_vocabulary']['related_words'].append({
                            "word": word_text, "url": safe_get_attr(word_link_tag, 'href')
                        })
    return entry_data


# --- Função Worker para o ThreadPoolExecutor ---
def worker_fetch_and_parse(word_key, gui_app):
    """Busca e analisa HTML para uma única palavra."""
    html_content = fetch_word_html(word_key, gui_app)
    if html_content:
        parsed_data = parse_cambridge_entry(html_content)
        if parsed_data and parsed_data.get("word"):
            return word_key, parsed_data, None  # Sucesso: (palavra, dados, None)
        else:
            error_detail = {"error": "Falha no parsing ou palavra não encontrada na página.",
                            "original_query": word_key, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
            return word_key, None, error_detail  # Erro de Parse: (palavra, None, erro)
    else:
        error_detail = {"error": "Falha ao buscar o conteúdo HTML.",
                        "original_query": word_key, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
        return word_key, None, error_detail  # Erro de Fetch: (palavra, None, erro)

# --- Lógica de Scraping (Modificada para Paralelismo) ---
def scraping_logic_thread(gui_app, initial_words, stop_event):
    all_words_data = load_existing_data(DATA_FILE, gui_app)
    processed_or_in_queue_set = set(all_words_data.keys())
    
    word_processing_queue = collections.deque()
    for word in initial_words:
        normalized_word = word.lower().strip()
        if normalized_word:
            word_processing_queue.append(normalized_word)
            # Adiciona ao set aqui também, para que se estiver na lista inicial mas já processado,
            # não seja adicionado à fila de processamento real abaixo se já existir em all_words_data.
            # A lógica de pular abaixo cuidará disso, mas é bom ter o set consistente.
            processed_or_in_queue_set.add(normalized_word)


    gui_app.log_message(f"--- Iniciando Coleta Paralela (Max Workers: {MAX_WORKERS}) ---")
    gui_app.log_message(f"Carregados {len(all_words_data)} registros de '{DATA_FILE}'.")
    gui_app.log_message(f"Fila inicial com {len(word_processing_queue)} palavras.")

    words_newly_collected_this_session = 0
    words_skipped_this_session = 0
    words_failed_this_session = 0
    
    # Usar um lock para salvar o arquivo, para o caso de querermos salvar mais frequentemente de dentro do loop as_completed
    save_data_lock = threading.Lock()

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        active_futures = {} # Mapeia future -> normalized_key

        while (word_processing_queue or active_futures) and not stop_event.is_set():
            # Submeter novas tarefas se houver palavras na fila e capacidade no executor
            # Limitar o número de futures submetidos de uma vez para evitar sobrecarregar a memória se o processamento de resultados for lento
            # Ou, como aqui, submeter enquanto a fila tiver itens e houver "espaço mental" para gerenciar futures
            while word_processing_queue and len(active_futures) < MAX_WORKERS * 2 and not stop_event.is_set() : # Ex: não mais que o dobro de workers em voo
                current_word_to_process = word_processing_queue.popleft()
                normalized_key = current_word_to_process.lower().strip() # Normalização já feita ao adicionar

                if not normalized_key: continue

                # Atualiza estatísticas antes de checar se pula (para refletir a fila diminuindo)
                gui_app.update_stats(
                    len(all_words_data) - words_failed_this_session,
                    words_skipped_this_session,
                    words_failed_this_session,
                    len(word_processing_queue) + len(active_futures) + 1 # +1 para o que está prestes a ser submetido/pulado
                )

                if normalized_key in all_words_data and all_words_data[normalized_key].get("word"):
                    # gui_app.log_message(f"'{normalized_key}' já processada. Pulando submissão.")
                    words_skipped_this_session += 1
                    continue
                elif normalized_key in all_words_data and "error" in all_words_data[normalized_key]:
                    # gui_app.log_message(f"'{normalized_key}' erro anterior. Pulando submissão.")
                    words_skipped_this_session += 1
                    continue
                
                # Se a palavra já foi adicionada ao processed_or_in_queue_set mas não está em all_words_data
                # (significa que está na fila mas ainda não foi submetida, ou foi submetida e está em active_futures)
                # A checagem `normalized_key in all_words_data` acima lida com o caso de já ter sido persistida.
                # Para evitar submeter a mesma palavra múltiplas vezes se ela for adicionada à fila várias vezes rapidamente:
                # O `processed_or_in_queue_set` já deveria conter `normalized_key` se ela foi pega da fila.
                # Se ela foi pulada acima, está OK. Se não, será submetida.
                
                gui_app.log_message(f"Submetendo: '{normalized_key}'...")
                future = executor.submit(worker_fetch_and_parse, normalized_key, gui_app)
                active_futures[future] = normalized_key

            if not active_futures and not word_processing_queue and not stop_event.is_set(): # Fila e workers ociosos
                gui_app.log_message("Fila de processamento vazia e nenhum worker ativo.")
                # Opcional: adicionar uma pequena pausa aqui se for esperado que a fila seja repopulada
                # time.sleep(0.5) # Se não houver mais futures, o loop as_completed abaixo não bloqueará
                # break # Ou sair se a intenção é terminar quando a fila inicial se esgota e nada mais é adicionado

            # Processar resultados dos futures que completaram
            # O timeout em as_completed permite que o loop verifique stop_event e submeta novas tasks
            # se o processamento de futures for lento.
            results_processed_in_batch = 0
            for future in as_completed(list(active_futures.keys()), timeout=0.5): # timeout pequeno para responsividade
                if stop_event.is_set(): break

                normalized_key_completed = active_futures.pop(future)
                results_processed_in_batch +=1
                
                try:
                    _word_key_returned, parsed_data_result, error_detail_result = future.result()
                    # _word_key_returned deve ser igual a normalized_key_completed

                    if parsed_data_result:
                        all_words_data[normalized_key_completed] = parsed_data_result
                        gui_app.log_message(f"Resultado OK para: '{normalized_key_completed}'.")
                        words_newly_collected_this_session += 1

                        # Adicionar palavras do SMART Vocabulary à fila
                        if parsed_data_result.get("smart_vocabulary", {}).get("related_words"):
                            new_smart_count = 0
                            for dict_word_info in parsed_data_result["smart_vocabulary"]["related_words"]:
                                url_smart = dict_word_info.get("url")
                                if url_smart:
                                    # Use sua função _extract_keyword_from_url aqui
                                    # Lembre-se que _extract_keyword_from_url precisa de REQUEST_BASE_URL_DICTIONARY
                                    potential_new_key = _extract_keyword_from_url(url_smart, REQUEST_BASE_URL_DICTIONARY)
                                    if potential_new_key and potential_new_key not in processed_or_in_queue_set:
                                        processed_or_in_queue_set.add(potential_new_key)
                                        word_processing_queue.append(potential_new_key)
                                        new_smart_count += 1
                            if new_smart_count > 0:
                                gui_app.log_message(f"+{new_smart_count} palavras do SMART Vocab para '{normalized_key_completed}' adicionadas à fila.")
                    
                    elif error_detail_result:
                        all_words_data[normalized_key_completed] = error_detail_result
                        gui_app.log_message(f"Resultado com ERRO para '{normalized_key_completed}': {error_detail_result.get('error')}")
                        words_failed_this_session += 1
                    
                except Exception as exc: # Erro ao obter resultado do future (ex: exceção no worker não capturada)
                    gui_app.log_message(f"Exceção no worker para '{normalized_key_completed}': {exc}")
                    all_words_data[normalized_key_completed] = {
                        "error": f"Exceção no worker: {str(exc)}",
                        "original_query": normalized_key_completed, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
                    }
                    words_failed_this_session += 1
                finally:
                    # Salvar após cada resultado processado, protegido por lock
                    with save_data_lock:
                        save_data(all_words_data, DATA_FILE, gui_app)
                    # O log de "Arquivo atualizado" pode ser muito frequente aqui, talvez logar a cada N salvamentos.

            if results_processed_in_batch > 0:
                 gui_app.log_message(f"Lote de {results_processed_in_batch} resultados processado. Arquivo salvo.")


            if stop_event.is_set():
                gui_app.log_message("Sinal de parada detectado, finalizando submissão e processamento de resultados.")
                # Limpar futures restantes se necessário, ou apenas deixar o 'with executor' lidar com o shutdown
                for future in list(active_futures.keys()): # Tenta cancelar futures que não iniciaram (melhor esforço)
                    future.cancel()
                break
            
            # Se não há mais palavras na fila e nenhuma tarefa ativa, o trabalho terminou.
            if not word_processing_queue and not active_futures:
                gui_app.log_message("Fila de processamento e tarefas ativas concluídas.")
                break
            
            # Pequena pausa para o loop principal não consumir 100% CPU se estiver apenas esperando por as_completed com timeout
            # time.sleep(0.1) # O timeout do as_completed já faz isso.

    # Fim do 'with ThreadPoolExecutor'
    if stop_event.is_set():
        gui_app.log_message("Coleta interrompida (final do executor).")
    
    gui_app.log_message("\n--- Coleta Finalizada (Sessão) ---")
    gui_app.log_message(f"Palavras novas coletadas: {words_newly_collected_this_session}")
    gui_app.log_message(f"Palavras puladas: {words_skipped_this_session}")
    gui_app.log_message(f"Falhas na coleta: {words_failed_this_session}")
    gui_app.log_message(f"Total de registros em '{DATA_FILE}': {len(all_words_data)}")
    gui_app.enable_start_button()

def _extract_keyword_from_url(url_string: str | None, base_url: str) -> str | None:
    """
    Helper function to extract a keyword from a dictionary URL.
    Assumes the keyword is the path segment after the base_url and before any query parameters.
    Example: "https://.../english/my-keyword?topic=..." -> "my-keyword"
    """
    if not url_string:
        return None
    
    url_string = url_string.strip()
    if url_string.startswith(base_url):
        # Get the part of the URL after the base_url
        path_after_base = url_string[len(base_url):]
        # The keyword is the part before any query parameters (e.g., '?topic=...')
        keyword = path_after_base.split('?')[0]
        return keyword if keyword else None # Ensure non-empty keyword
    return None

def find_new_related_keywords(
    data_store: dict[str, dict], 
    dictionary_base_url: str = REQUEST_BASE_URL_DICTIONARY
) -> set[str]:
    """
    Scans a data store of word entries, extracts related keywords from their
    'smart_vocabulary' URLs, and returns a set of unique related keywords
    that are not already present as top-level keys in the data_store.

    Args:
        data_store (dict[str, dict]): The main dictionary where keys are existing words
                                      (keywords) and values are their detailed data.
                                      It's expected that entry values are dictionaries
                                      which might contain a 'smart_vocabulary' key,
                                      which in turn might contain a 'related_words' list.
        dictionary_base_url (str): The base URL prefix for dictionary entries,
                                   used to extract keywords from related word URLs.

    Returns:
        set[str]: A set of unique related keywords (strings) found from the URLs,
                  which are not already keys in the input data_store.
                  Keywords are in the format like 'anecdote', 'be-another-story'.
    """
    all_extracted_related_keywords = set()
    
    # Iterate through the values (word data entries) in the data store
    for entry_data in data_store.values():
        # Ensure the entry_data itself is a dictionary to safely use .get()
        if not isinstance(entry_data, dict):
            continue 
            
        # Safely navigate to the 'related_words' list
        smart_vocabulary_data = entry_data.get("smart_vocabulary", {})
        if not isinstance(smart_vocabulary_data, dict): # Ensure smart_vocabulary_data is a dict
            continue
            
        related_word_info_list = smart_vocabulary_data.get("related_words", [])
        if not isinstance(related_word_info_list, list): # Ensure related_word_info_list is a list
            continue

        for related_info_item in related_word_info_list:
            # Ensure the item within 'related_words' is a dictionary
            if not isinstance(related_info_item, dict):
                continue

            url = related_info_item.get("url")
            extracted_keyword = _extract_keyword_from_url(url, dictionary_base_url)
            
            if extracted_keyword:
                all_extracted_related_keywords.add(extracted_keyword)
    
    # Get the set of keywords already present in the data_store
    existing_keywords_in_store = set(data_store.keys())
    
    # Find which of the extracted related keywords are new
    # (i.e., not already in the data_store keys)
    newly_discovered_keywords = all_extracted_related_keywords.difference(existing_keywords_in_store)
    
    return newly_discovered_keywords


def read_json(path_json):
    try:
        with open(path_json, "r", encoding="utf-8") as file:
            json_data = json.load(file)
        return json_data
    except FileNotFoundError:
        print(f"Erro: O arquivo '{DATA_FILE}' não foi encontrado.")
        return {}
    except json.JSONDecodeError:
        print(f"Erro: O arquivo '{DATA_FILE}' não contém um JSON válido ou está corrompido.")
        return {}
    except Exception as e:
        print(f"Ocorreu um erro inesperado ao ler o arquivo: {e}")
        return {}


# --- Classe da Interface Gráfica Tkinter ---
class ScraperAppGUI:
    def __init__(self, master_root):
        self.master = master_root
        master_root.title("Cambridge Dictionary Scraper")
        master_root.geometry("700x550")

        self.stop_event = threading.Event()
        self.scraper_thread = None

        # Frame para controles
        control_frame = ttk.Frame(master_root, padding="10")
        control_frame.pack(fill=tk.X)

        ttk.Label(control_frame, text="Palavras Iniciais (separadas por vírgula):").pack(side=tk.LEFT, padx=(0, 5))
        self.words_entry = ttk.Entry(control_frame, width=40)
        self.words_entry.insert(0, "story, have, elegance, ubiquitous, nonexistentwordxyz")
        self.words_entry.pack(side=tk.LEFT, expand=True, fill=tk.X, padx=5)
        
        self.start_button = ttk.Button(control_frame, text="Iniciar Coleta", command=self.start_scraping)
        self.start_button.pack(side=tk.LEFT, padx=5)

        self.stop_button = ttk.Button(control_frame, text="Parar", command=self.stop_scraping, state=tk.DISABLED)
        self.stop_button.pack(side=tk.LEFT)

        # Área de Log
        log_frame = ttk.LabelFrame(master_root, text="Log de Atividades", padding="10")
        log_frame.pack(expand=True, fill=tk.BOTH, padx=10, pady=5)
        
        self.log_text_area = scrolledtext.ScrolledText(log_frame, wrap=tk.WORD, height=15, state=tk.DISABLED)
        self.log_text_area.pack(expand=True, fill=tk.BOTH)

        # Frame para Estatísticas
        stats_frame = ttk.LabelFrame(master_root, text="Estatísticas", padding="10")
        stats_frame.pack(fill=tk.X, padx=10, pady=(0,10))

        self.stats_label = ttk.Label(stats_frame, text="Coletadas: 0 | Puladas: 0 | Falhas: 0 | Fila: 0")
        self.stats_label.pack()
        
        master_root.protocol("WM_DELETE_WINDOW", self.on_closing)

    def log_message(self, message):
        if hasattr(self, 'log_text_area') and self.log_text_area.winfo_exists():
            self.log_text_area.config(state=tk.NORMAL)
            self.log_text_area.insert(tk.END, f"{time.strftime('%H:%M:%S')} - {message}\n")
            self.log_text_area.see(tk.END)
            self.log_text_area.config(state=tk.DISABLED)
            self.master.update_idletasks() # Força atualização da UI

    def update_stats(self, collected, skipped, failed, queue_size):
        if hasattr(self, 'stats_label') and self.stats_label.winfo_exists():
            self.stats_label.config(text=f"Coletadas (total): {collected} | Puladas (sessão): {skipped} | Falhas (sessão): {failed} | Fila: {queue_size}")
            self.master.update_idletasks()

    def start_scraping(self):
        initial_words_str = self.words_entry.get()

        if not initial_words_str.strip():
            messagebox.showwarning("Entrada Inválida", "Por favor, insira algumas palavras iniciais.")
            return
            
        initial_words = [word.strip() for word in initial_words_str.split(',') if word.strip()]

        json_data = read_json(path_json=DATA_FILE)

        new_keywords_to_fetch = find_new_related_keywords(json_data)


        initial_words.extend(new_keywords_to_fetch)
        
        if not initial_words:
            messagebox.showwarning("Entrada Inválida", "Nenhuma palavra válida para processar após limpeza.")
            return

        self.log_text_area.config(state=tk.NORMAL)
        self.log_text_area.delete('1.0', tk.END) # Limpa log anterior
        self.log_text_area.config(state=tk.DISABLED)

        self.stop_event.clear()
        self.start_button.config(state=tk.DISABLED)
        self.stop_button.config(state=tk.NORMAL)
        self.words_entry.config(state=tk.DISABLED)
        
        self.log_message(f"Iniciando coleta para: {', '.join(initial_words)}")
        
        # Passa a instância da GUI e o evento de parada para a thread
        self.scraper_thread = threading.Thread(target=scraping_logic_thread, args=(self, initial_words, self.stop_event))
        self.scraper_thread.daemon = True 
        self.scraper_thread.start()
        
        # Não é mais necessário, pois a thread chama enable_start_button() no final
        # self.master.after(100, self.check_thread_status)

    def stop_scraping(self):
        if self.scraper_thread and self.scraper_thread.is_alive():
            self.stop_event.set()
            self.log_message("Sinal de parada enviado à thread de coleta...")
        self.stop_button.config(state=tk.DISABLED) # Desabilita imediatamente

    def enable_start_button(self):
        """Chamado pela thread de scraping quando ela termina."""
        if self.master.winfo_exists(): # Verifica se a janela ainda existe
            self.start_button.config(state=tk.NORMAL)
            self.stop_button.config(state=tk.DISABLED)
            self.words_entry.config(state=tk.NORMAL)
            self.log_message("Pronto para nova coleta ou fechar.")
            
    def on_closing(self):
        if self.scraper_thread and self.scraper_thread.is_alive():
            self.log_message("Tentando parar a coleta antes de fechar...")
            self.stop_event.set()
            # Poderia esperar um pouco pela thread aqui, ou apenas avisar
            if messagebox.askokcancel("Sair", "A coleta de dados está em andamento. Deseja realmente sair? O progresso atual foi salvo."):
                self.master.destroy()
            else:
                return # Não fecha
        else:
            self.master.destroy()

# --- Ponto de Entrada Principal ---
if __name__ == "__main__":
    if parse_cambridge_entry.__doc__ and "placeholder" in parse_cambridge_entry.__doc__:
         print("ERRO: A função parse_cambridge_entry está incompleta. Copie a versão completa.")
         exit()

    root = tk.Tk()
    app = ScraperAppGUI(root)
    root.mainloop()

In [51]:
import json 

def read_json(path_json):
    try:
        with open(path_json, "r", encoding="utf-8") as file:
            json_data = json.load(file)
        return json_data
    except FileNotFoundError:
        print(f"Erro: O arquivo '{DATA_FILE}' não foi encontrado.")
        return {}
    except json.JSONDecodeError:
        print(f"Erro: O arquivo '{DATA_FILE}' não contém um JSON válido ou está corrompido.")
        return {}
    except Exception as e:
        print(f"Ocorreu um erro inesperado ao ler o arquivo: {e}")
        return {}

json_data = read_json(path_json=DATA_FILE)

In [52]:
import pandas as pd

pd.DataFrame(json_data).T.reset_index().shape

(529, 10)

In [ ]:
def find_keys_without_word_attribute(json_data: dict) -> list:
    """
    Filtra um dicionário e retorna uma lista de chaves cujos valores
    não são dicionários contendo uma chave "word" com um valor "truthy".

    Uma chave é incluída se o valor associado:
    - Não é um dicionário.
    - É um dicionário, mas não possui a chave "word".
    - É um dicionário, possui a chave "word", mas o valor dessa chave é "falsey"
      (ex: None, string vazia, False, 0, lista/dicionário vazio).

    Args:
        dados_json: O dicionário a ser verificado.

    Returns:
        Uma lista de chaves que satisfazem a condição.
    """
    keys_without_word_attribute = [
        chave for chave, valor in json_data.items()
        if not (isinstance(valor, dict) and valor.get("word"))
    ]
    return keys_without_word_attribute

find_keys_without_word_attribute(json_data)

['nonexistentwordxyz',
 'be-another-story',
 'gussy-up',
 'from-pillar-to-post',
 'from-top-to-bottom',
 'from-top-to-toe',
 'hunt-search-high-and-low',
 'on-at-every-corner',
 'for-good-measure',
 'be-all-part-of-life-s-rich-tapestry-pageant']

In [ ]:
DEFAULT_CAMBRIDGE_BASE_URL = "https://dictionary.cambridge.org/dictionary/english/"



len(dados)

1063

In [43]:
list_of_words_in_database = json_data.keys()

set(['story', 'have', 'elegance', "degdue"]).difference(set(list_of_words_in_database))

{'degdue'}

In [40]:
list_of_words_in_database

dict_keys(['story', 'have', 'elegance', 'ubiquitous', 'nonexistentwordxyz', 'anecdote', 'another', 'anti-narrative', 'backstory', 'be-another-story', 'bodice-ripper', 'cautionary-tale', 'commentary', 'horror-story', 'in-medias-res', 'kompromat', 'legendary', 'lore', 'rundown', 'running-commentary', 'scenario', 'semi-legendary', 'shaggy-dog-story', 'strand', 'write-up', 'aesthetic', 'attractiveness', 'beauty', 'chic', 'daintiness', 'grace', 'gracefulness', 'grandeur', 'gussy-up', 'heart-stopping', 'magnificence', 'majestically', 'majesty', 'nobility', 'picturesqueness', 'stateliness', 'sublimity', 'sultriness', 'swoony', 'va-va-voom', 'across', 'all-around', 'all-over', 'anyplace', 'anywhere', 'coast', 'everywhere-else', 'from-pillar-to-post', 'from-top-to-bottom', 'from-top-to-toe', 'hunt-search-high-and-low', 'inch', 'on-at-every-corner', 'pan', 'pillar', 'round', 'someplace', 'somewhere', 'spread', 'throughout', 'broad-brushstrokes', 'misdescription', 'short-fiction', 'added', 'addit

In [33]:
json_data.get("story").get("smart_vocabulary").get("related_words")[0].get("url")

' https://dictionary.cambridge.org/dictionary/english/anecdote?topic=accounts-and-stories '

In [21]:
keys_without_word_attribute = [
        chave for chave, valor in json_data.items()
        if (isinstance(valor, dict) and valor.get("smart_vocabulary"))
    ]

keys_without_word_attribute

['story',
 'have',
 'elegance',
 'ubiquitous',
 'anecdote',
 'another',
 'anti-narrative',
 'backstory',
 'bodice-ripper',
 'cautionary-tale',
 'commentary',
 'horror-story',
 'in-medias-res',
 'kompromat',
 'legendary',
 'lore',
 'rundown',
 'running-commentary',
 'scenario',
 'semi-legendary',
 'shaggy-dog-story',
 'strand',
 'write-up',
 'aesthetic',
 'attractiveness',
 'beauty',
 'chic',
 'daintiness',
 'grace',
 'gracefulness',
 'grandeur',
 'heart-stopping',
 'magnificence',
 'majestically',
 'majesty',
 'nobility',
 'picturesqueness',
 'stateliness',
 'sublimity',
 'sultriness',
 'swoony',
 'va-va-voom',
 'across',
 'all-around',
 'all-over',
 'anyplace',
 'anywhere',
 'coast',
 'everywhere-else',
 'inch',
 'pan',
 'pillar',
 'round',
 'someplace',
 'somewhere',
 'spread',
 'throughout',
 'broad-brushstrokes',
 'misdescription',
 'short-fiction',
 'added',
 'additional',
 'additionally',
 'again',
 'along',
 'et-cetera',
 'etc',
 'excess',
 'filler',
 'premium',
 'rate',
 'regar

In [1]:
import requests
from bs4 import BeautifulSoup
from typing import List, Dict, Optional

class CambridgeScraper:
    """
    Um scraper BÁSICO e demonstração para o Cambridge Dictionary.
    NÃO RECOMENDADO para uso em produção devido à fragilidade e termos de uso.
    """
    BASE_URL = "https://dictionary.cambridge.org/dictionary/english/"

    def get_word_definition(self, word: str) -> Optional[Dict]:
        """
        Tenta raspar a definição de uma palavra do Cambridge Dictionary.
        
        Args:
            word (str): A palavra a ser pesquisada.
            
        Returns:
            Optional[Dict]: Um dicionário com definições e exemplos, ou None.
        """
        url = f"{self.BASE_URL}{word.lower()}"
        try:
            response = requests.get(url, timeout=5)
            response.raise_for_status()
            
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # As classes HTML podem mudar! Isso é apenas um exemplo.
            definitions = []
            
            # Encontre o elemento principal de definição
            entry_body = soup.find('div', class_='entry-body')
            if not entry_body:
                # Tentar outra classe ou estrutura se a primeira falhar
                entry_body = soup.find('div', class_='pr entry-body__el')


            if entry_body:
                # Iterar sobre as definições
                for def_block in entry_body.find_all('div', class_='def-block'):
                    definition_text = def_block.find('div', class_='def').text.strip()
                    examples = []
                    for ex in def_block.find_all('div', class_='examp'):
                        examples.append(ex.text.strip())
                    definitions.append({
                        "definition": definition_text,
                        "examples": examples
                    })
            
            if definitions:
                return {"word": word, "definitions": definitions}
            
            return None # Nenhuma definição encontrada
            
        except requests.exceptions.RequestException as e:
            print(f"Erro ao acessar {url}: {e}")
            return None
        except AttributeError:
            print(f"Estrutura HTML diferente do esperado para '{word}'. O scraper pode precisar ser atualizado.")
            return None
        

# --- Exemplo de Uso do Scraper ---
if __name__ == "__main__":
    print("\n--- Usando o Web Scraper (Apenas para demonstração) ---")
    scraper = CambridgeScraper()
    
    word_to_scrape = "elegance"
    scraped_data = scraper.get_word_definition(word_to_scrape)
    
    if scraped_data:
        print(f"Definição raspada para '{scraped_data['word']}':")
        for def_entry in scraped_data['definitions']:
            print(f"  - {def_entry['definition']}")
            for example in def_entry['examples']:
                print(f"    Exemplo: {example}")
    else:
        print(f"Não foi possível raspar a definição para '{word_to_scrape}'.")


--- Usando o Web Scraper (Apenas para demonstração) ---
Erro ao acessar https://dictionary.cambridge.org/dictionary/english/elegance: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Não foi possível raspar a definição para 'elegance'.


In [ ]:
import requests
from bs4 import BeautifulSoup
import json # Para imprimir o dicionário de forma legível

# URL base para construir links completos de áudio e referências
BASE_URL_CAMBRIDGE = "https://dictionary.cambridge.org"

def safe_get_text(element, default=""):
    """Extrai o texto de um elemento BeautifulSoup de forma segura."""
    return element.get_text() if element else default

def safe_get_attr(element, attr, default=""):
    """Extrai um atributo de um elemento BeautifulSoup de forma segura."""
    return element.get(attr, default) if element else default

def parse_cambridge_entry(html_content):
    """
    Analisa o conteúdo HTML de uma entrada do Cambridge Dictionary e extrai os dados.
    """
    soup = BeautifulSoup(html_content, 'html.parser')
    entry_data = {
        "word": "",
        "part_of_speech": "",
        "grammar": "",
        "pronunciations": {"uk": {}, "us": {}},
        "senses": [],
        "smart_vocabulary": {"topic": {}, "related_words": []}
    }

    # Encontra o bloco principal da entrada da palavra
    # A classe 'pr entry-body__el' é a que você identificou no seu HTML
    entry_body = soup.find('div', class_='pr entry-body__el')
    if not entry_body:
        print("Erro: Bloco principal da entrada ('pr entry-body__el') não encontrado.")
        return entry_data # Retorna dados vazios ou parciais

    # --- Palavra, Classe Gramatical, Gramática ---
    pos_header = entry_body.find('div', class_='pos-header')
    if pos_header:
        headword_span = pos_header.find('span', class_='hw dhw') # Palavra principal
        entry_data['word'] = safe_get_text(headword_span)

        pos_span = pos_header.find('span', class_='pos dpos') # Classe gramatical (noun, verb, etc.)
        entry_data['part_of_speech'] = safe_get_text(pos_span)
        
        gram_span = pos_header.find('span', class_='gram dgram') # Informações gramaticais (ex: [U], [C])
        entry_data['grammar'] = safe_get_text(gram_span)

        # --- Pronúncias ---
        uk_pron_span = pos_header.find('span', class_='uk dpron-i')
        if uk_pron_span:
            uk_ipa = safe_get_text(uk_pron_span.find('span', class_='ipa dipa'))
            uk_audio_source = uk_pron_span.find('source', type='audio/mpeg')
            uk_audio_url = safe_get_attr(uk_audio_source, 'src')
            entry_data['pronunciations']['uk'] = {
                'ipa': uk_ipa,
                'audio': BASE_URL_CAMBRIDGE + uk_audio_url if uk_audio_url else ""
            }

        us_pron_span = pos_header.find('span', class_='us dpron-i')
        if us_pron_span:
            us_ipa = safe_get_text(us_pron_span.find('span', class_='ipa dipa'))
            us_audio_source = us_pron_span.find('source', type='audio/mpeg')
            us_audio_url = safe_get_attr(us_audio_source, 'src')
            entry_data['pronunciations']['us'] = {
                'ipa': us_ipa,
                'audio': BASE_URL_CAMBRIDGE + us_audio_url if us_audio_url else ""
            }
    else:
        # Tenta encontrar a palavra principal mesmo fora do pos-header (fallback)
        headword_fallback = entry_body.find('span', class_='hw dhw')
        if headword_fallback:
             entry_data['word'] = safe_get_text(headword_fallback)


    # --- Sentidos (definições, exemplos, etc.) ---
    pos_body = entry_body.find('div', class_='pos-body')
    if pos_body:
        # Cada 'def-block' geralmente representa um sentido ou um grupo de informações relacionadas
        for def_block in pos_body.find_all('div', class_='def-block ddef_block'):
            current_sense = {}

            # Nível CEFR (Common European Framework of Reference for Languages)
            ddef_h = def_block.find('div', class_='ddef_h')
            if ddef_h:
                def_info = ddef_h.find('span', class_='def-info')
                if def_info:
                    epp_xref = def_info.find('span', class_='epp-xref')
                    current_sense['cefr_level'] = safe_get_text(epp_xref)

            # Definição
            def_div = def_block.find('div', class_='def ddef_d db')
            current_sense['definition'] = safe_get_text(def_div)

            # Exemplos principais
            examples = []
            def_body_ddef_b = def_block.find('div', class_='def-body ddef_b')
            if def_body_ddef_b:
                for ex_div in def_body_ddef_b.find_all('div', class_='examp dexamp', recursive=False):
                    example_text = safe_get_text(ex_div.find('span', class_='eg deg'))
                    if example_text:
                        examples.append(example_text)
            current_sense['examples'] = examples
            
            # Mais exemplos (geralmente dentro de um acordeão)
            more_examples = []
            # O daccord pode ser filho direto do def_block ou dentro do sense_body
            daccord_more_examples = def_block.find('div', class_='daccord')
            if daccord_more_examples:
                 # Verifica se é o acordeão de "More examples"
                header_span = daccord_more_examples.find('span', class_='showmore')
                if header_span and header_span.get_text(strip=True) == "More examples":
                    for li_tag in daccord_more_examples.find_all('li', class_='eg dexamp hax'):
                        more_examples.append(safe_get_text(li_tag))
            current_sense['more_examples'] = more_examples

            # "See" (termos relacionados)
            see_also_terms = []
            if def_body_ddef_b:
                see_xref_div = def_body_ddef_b.find('div', class_='xref see hax dxref-w')
                if see_xref_div:
                    for item_div in see_xref_div.find_all('div', class_='item lc'):
                        term_span = item_div.find('span', class_='x-h dx-h')
                        term_text = safe_get_text(term_span)
                        term_link_tag = item_div.find('a')
                        term_url = safe_get_attr(term_link_tag, 'href')
                        if term_text:
                            see_also_terms.append({
                                "term": term_text,
                                "url": BASE_URL_CAMBRIDGE + term_url if term_url and not term_url.startswith('http') else term_url
                            })
            current_sense['see_also'] = see_also_terms
            
            if current_sense.get('definition') or current_sense.get('examples'): # Adiciona apenas se houver dados úteis
                entry_data['senses'].append(current_sense)

    # --- SMART Vocabulary ---
    # Localiza a seção SMART Vocabulary (pode estar em diferentes posições dependendo da página)
    smart_vocab_div = entry_body.find('div', class_='smartt daccord')
    if smart_vocab_div:
        topic_link_tag = smart_vocab_div.find('div', class_='daccord_lt')
        if topic_link_tag:
            topic_anchor = topic_link_tag.find('a')
            entry_data['smart_vocabulary']['topic'] = {
                "name": safe_get_text(topic_anchor),
                "url": safe_get_attr(topic_anchor, 'href') # URLs aqui geralmente são absolutos
            }
            
        related_words_list = smart_vocab_div.find('ul', class_='hul-u')
        if related_words_list:
            for li_tag in related_words_list.find_all('li', class_='lc'):
                word_link_tag = li_tag.find('a')
                if word_link_tag:
                    word_text = ""
                    # Tenta extrair o texto de forma mais limpa, lidando com estruturas internas
                    base_span = word_link_tag.find('span', class_='base')
                    if base_span:
                        text_parts = [s.get_text(strip=True) for s in base_span.find_all(True, recursive=False) if s.get_text(strip=True)]
                        word_text = ' '.join(text_parts)
                        if not word_text: # Fallback se a estrutura interna for diferente
                            word_text = safe_get_text(base_span)
                    else: # Fallback para o texto completo do link se 'span.base' não existir
                        results_span = word_link_tag.find('span', class_='results')
                        if results_span:
                            word_text = safe_get_text(results_span)
                        else:
                            word_text = safe_get_text(word_link_tag) # Último fallback

                    word_url = safe_get_attr(word_link_tag, 'href')
                    if word_text:
                        entry_data['smart_vocabulary']['related_words'].append({
                            "word": word_text,
                            "url": word_url # URLs aqui geralmente são absolutos
                        })
                        
    return entry_data

# --- Seu código para fazer a requisição HTTP ---
REQUEST_BASE_URL = "https://dictionary.cambridge.org/dictionary/english/"
word_to_search = "elegance"
url_to_fetch = f"{REQUEST_BASE_URL}{word_to_search.lower()}"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "Accept-Language": "en-US,en;q=0.9,pt-BR;q=0.8,pt;q=0.7",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
}

html_page_content = ""

try:
    print(f"Tentando acessar: {url_to_fetch} com User-Agent customizado...")
    # response = requests.get(url_to_fetch, headers=headers, timeout=15) # Timeout aumentado
    # response.raise_for_status() 

    print(f"Status Code: {response.status_code}")
    # print(f"Tamanho da Resposta (chars): {len(response.text)}")
    # print(response.text[:2000]) # Imprime uma parte maior para depuração inicial
    print("Acesso bem-sucedido! O conteúdo da página foi recebido.")
    html_page_content = response.text

except requests.exceptions.ConnectionError as e:
    print(f"Erro de Conexão: O servidor pode ter fechado a conexão ou recusado. {e}")
    print("Isso pode indicar que o servidor está bloqueando o request, mesmo com User-Agent.")
except requests.exceptions.Timeout:
    print("Erro de Timeout: A requisição demorou muito para responder.")
except requests.exceptions.HTTPError as e:
    print(f"Erro HTTP: {e.response.status_code} - {e.response.reason}")
    print("O conteúdo da página pode não ser o esperado ou pode ser uma página de erro.")
    html_page_content = e.response.text # Algumas vezes, a página de erro ainda contém HTML útil para análise do bloqueio
except requests.exceptions.RequestException as e:
    print(f"Ocorreu um erro ao fazer a requisição: {e}")
    print("Verifique se o URL está correto e se o site está online.")
except Exception as e:
    print(f"Um erro inesperado ocorreu durante a requisição: {e}")


# --- Processamento do HTML ---
if html_page_content:
    print("\nIniciando o parsing do HTML...")
    extracted_data = parse_cambridge_entry(html_page_content)
    
    print("\n--- Dados Extraídos ---")
    # Usando json.dumps para uma saída formatada e legível do dicionário
    print(json.dumps(extracted_data, indent=4, ensure_ascii=False))
else:
    print("\nNenhum conteúdo HTML para analisar devido a erro na requisição.")
    print("Se desejar testar o parser com o HTML fornecido diretamente, descomente o bloco abaixo.")

    # --- Bloco para testar o parser com o HTML fornecido diretamente ---
    # print("\n--- Testando o parser com o HTML fornecido ---")
    # provided_html_snippet = """
    # <div class="pr entry-body__el"> ... (COLE SEU SNIPPET HTML AQUI) ... </div>
    # """ # Substitua pelo seu snippet completo
    # # Verifique se o snippet está completo e correto.
    # # Se o snippet for muito grande, pode ser melhor lê-lo de um arquivo.
    # if "COLE SEU SNIPPET HTML AQUI" in provided_html_snippet:
    #     print("Por favor, substitua o placeholder com seu snippet HTML para testar o parser.")
    # else:
    #     extracted_data_from_snippet = parse_cambridge_entry(provided_html_snippet)
    #     print("\n--- Dados Extraídos do Snippet ---")
    #     print(json.dumps(extracted_data_from_snippet, indent=4, ensure_ascii=False))

Tentando acessar: https://dictionary.cambridge.org/dictionary/english/elegance com User-Agent customizado...
Status Code: 200
Acesso bem-sucedido! O conteúdo da página foi recebido.

Iniciando o parsing do HTML...
def_div:  <div class="def ddef_d db">the <a class="query" href="https://dictionary.cambridge.org/dictionary/english/quality" rel="" title="quality">quality</a> of being <a class="query" href="https://dictionary.cambridge.org/dictionary/english/grace" rel="" title="graceful">graceful</a> and <a class="query" href="https://dictionary.cambridge.org/dictionary/english/attractive" rel="" title="attractive">attractive</a> in <a class="query" href="https://dictionary.cambridge.org/dictionary/english/appearance" rel="" title="appearance">appearance</a> or <a class="query" href="https://dictionary.cambridge.org/dictionary/english/behaviour" rel="" title="behaviour">behaviour</a>: </div>

--- Dados Extraídos ---
{
    "word": "elegance",
    "part_of_speech": "noun",
    "grammar": "[ 

In [19]:
import pandas as pd

pd.DataFrame([extracted_data])

,word,part_of_speech,grammar,pronunciations,senses,smart_vocabulary
0,elegance,noun,[ U ],"{'uk': {'ipa': '', 'audio': 'https://dictionar...","[{'cefr_level': 'C1', 'definition': 'the quali...",{'topic': {'name': 'Beauty and attractiveness'...


In [13]:
from bs4 import BeautifulSoup
from typing import List, Dict, Optional

class CambridgeDefinitionExtractor:
    """
    Uma classe elegante para extrair informações abrangentes de uma entrada do Cambridge Dictionary.
    Foca na robustez e extração granular de dados.
    """
    
    def __init__(self, html_content: str):
        """
        Inicializa o extrator com o conteúdo HTML da página.
        
        Args:
            html_content (str): O HTML completo da página de definição da palavra.
        """
        self.soup = BeautifulSoup(html_content, 'html.parser')

    def extract_all_data(self) -> Dict:
        """
        Extrai todos os dados relevantes de uma entrada do Cambridge Dictionary.
        
        Retorna:
            Dict: Um dicionário contendo todas as informações extraídas,
                  incluindo seções de significados, pronúncias, etc.
        """
        data = {
            "word_title": self._extract_word_title(),
            "pronunciations": self._extract_pronunciations(),
            "meanings": self._extract_meanings(),
            "phrasal_verbs_and_idioms": self._extract_phrasal_verbs_and_idioms(),
            "related_expressions": self._extract_related_expressions()
        }
        return data

    def _extract_word_title(self) -> Optional[str]:
        """Extrai o título principal da palavra."""
        # Geralmente a palavra principal aparece em um h2 ou span com uma classe específica
        title_element = self.soup.find('h2', class_='clm-font-5-responsive') or \
                        self.soup.find('div', class_='hw pos-header') # Tenta classes comuns
        if title_element:
            # Pega o texto antes de qualquer span de pronúncia ou categoria
            return title_element.find('span', class_='headword').text.strip() if title_element.find('span', class_='headword') else title_element.text.strip()
        return None

    def _extract_pronunciations(self) -> List[Dict]:
        """Extrai as informações de pronúncia (áudio e texto fonético)."""
        pronunciations = []
        # Procurar por elementos que contenham informações de pronúncia
        # As classes comuns são 'pron', 'us', 'uk', 'ipa' e áudios em 'audio_play_button'
        
        # Pode haver várias seções de pronúncia (US, UK, etc.)
        for pron_block in self.soup.find_all('span', class_=['pron-info', 'di-info', 'dpron-i']):
            pron_text_elem = pron_block.find('span', class_=['ipa', 'dipa'])
            pron_text = pron_text_elem.text.strip() if pron_text_elem else None
            
            audio_elem = pron_block.find('source', type='audio/mpeg')
            audio_url = audio_elem['src'] if audio_elem else None

            region_elem = pron_block.find('span', class_=['region', 'dregion'])
            region = region_elem.text.strip() if region_elem else None # UK, US

            if pron_text or audio_url:
                pronunciations.append({
                    "region": region,
                    "ipa": pron_text,
                    "audio_url": audio_url
                })
        return pronunciations

    def _extract_meanings(self) -> List[Dict]:
        """
        Extrai as diferentes seções de significado (blocos de definição).
        Cada significado pode ter categoria gramatical, definições, exemplos.
        """
        meanings = []
        # O corpo principal das definições é geralmente encapsulado em 'entry-body' ou 'pr entry-body__el'
        main_entry_body = self.soup.find('div', class_='entry-body')
        if not main_entry_body:
            main_entry_body = self.soup.find('div', class_='pr entry-body__el')
        
        if not main_entry_body:
            return meanings

        # Iterar sobre as seções de definição, que podem ser agrupadas por categoria (e.g., 'noun', 'verb')
        # As classes como 'pos-header' ou 'sense-block' são bons indicadores
        for pos_block in main_entry_body.find_all('div', class_='pos-body'): # Principal bloco por parte da fala
            pos_tag_elem = pos_block.find('span', class_=['posgram', 'pos'])
            pos_tag = pos_tag_elem.text.strip() if pos_tag_elem else "N/A" # Ex: noun, verb

            for sense_block in pos_block.find_all('div', class_='sense-block'):
                # Código de referência ou número da definição
                guideword_elem = sense_block.find('span', class_=['guideword', 'dsense_intro'])
                guideword = guideword_elem.text.strip() if guideword_elem else None

                # Extrair definições e exemplos dentro deste bloco de sentido
                definitions_list = []
                for def_block in sense_block.find_all('div', class_=['def-block', 'ddef_block']):
                    definition_text_elem = def_block.find('div', class_=['def', 'ddef_d'])
                    definition_text = definition_text_elem.text.strip() if definition_text_elem else "Definição não encontrada"
                    
                    # Remover o prefixo de categoria (e.g., '[ C ]') se presente
                    definition_text = self._clean_definition_text(definition_text)

                    examples = [ex.text.strip() for ex in def_block.find_all('div', class_=['examp', 'dexample'])]
                    
                    # Sinônimos e Antônimos
                    syn_ant_block = def_block.find('div', class_='synonyms') or def_block.find('div', class_='antonyms')
                    synonyms = []
                    antonyms = []
                    if syn_ant_block:
                        for item in syn_ant_block.find_all('a', class_='syn-link'):
                            synonyms.append(item.text.strip())
                        for item in syn_ant_block.find_all('a', class_='ant-link'):
                            antonyms.append(item.text.strip())


                    definitions_list.append({
                        "text": definition_text,
                        "examples": examples,
                        "synonyms": synonyms,
                        "antonyms": antonyms
                    })
                
                if definitions_list:
                    meanings.append({
                        "part_of_speech": pos_tag,
                        "guideword": guideword,
                        "definitions": definitions_list
                    })
        return meanings
    
    def _clean_definition_text(self, text: str) -> str:
        """Remove padrões indesejados do texto da definição, como '[ C ]' ou '[ T ]'."""
        import re
        # Expressão regular para remover padrões como '[ C ]', '[ T ]', '[ S ]', '[ U ]' no início da string
        # e também os elementos de áudio play button que podem estar dentro da definição.
        cleaned_text = re.sub(r'\[\s*[CSUTILB]\s*\]', '', text).strip()
        # Remove texto de botões de áudio que porventura sejam capturados
        cleaned_text = re.sub(r'[\s\S]*?audio_play_button\s*', '', cleaned_text)
        return cleaned_text


    def _extract_phrasal_verbs_and_idioms(self) -> List[Dict]:
        """Extrai phrasal verbs e expressões idiomáticas associadas à palavra."""
        expressions = []
        # As expressões podem estar em seções como 'idm-block' ou 'pv-block'
        for expr_block in self.soup.find_all('div', class_=['idm-block', 'pv-block']):
            phrase_elem = expr_block.find('span', class_=['phrase-title', 'di-title'])
            phrase = phrase_elem.text.strip() if phrase_elem else None
            
            if phrase:
                # Extrair definições e exemplos para a frase/idioma
                expr_definitions = []
                for def_block in expr_block.find_all('div', class_=['def-block', 'ddef_block']):
                    definition_text_elem = def_block.find('div', class_=['def', 'ddef_d'])
                    definition_text = definition_text_elem.text.strip() if definition_text_elem else "Definição não encontrada"
                    definition_text = self._clean_definition_text(definition_text)
                    examples = [ex.text.strip() for ex in def_block.find_all('div', class_=['examp', 'dexample'])]
                    
                    expr_definitions.append({
                        "text": definition_text,
                        "examples": examples
                    })
                
                if expr_definitions:
                    expressions.append({
                        "phrase": phrase,
                        "definitions": expr_definitions
                    })
        return expressions
    
    def _extract_related_expressions(self) -> List[Dict]:
        """Extrai expressões e palavras relacionadas (ex: "Related words" ou "More examples")."""
        related_expressions = []
        # Elementos comuns como 'related-words' ou 'more-examples'
        for related_block in self.soup.find_all('div', class_=['related-words', 'other-related-items']):
            title_elem = related_block.find('h3', class_='cdo-section-title') # Ou outra tag de título
            title = title_elem.text.strip() if title_elem else "Expressões Relacionadas"
            
            # Pegar links ou textos de expressões
            items = [li.text.strip() for li in related_block.find_all('li', class_='cdo-usage-item')]
            
            if items:
                related_expressions.append({
                    "section_title": title,
                    "items": items
                })
        return related_expressions


# --- Exemplo de Uso (Assumindo que você já tem o HTML) ---
if __name__ == "__main__":
    importante: O 'html_content' deve vir de uma requisição bem-sucedida,
    # seja com headers (User-Agent) ou Selenium, como discutimos.
    # Exemplo: html_content = response.text
    
    # Para demonstração, estou usando um placeholder.
    # Na prática, você faria a requisição HTTP aqui.
    print("Por favor, certifique-se de que o 'html_content' é o HTML completo da página.")
    print("Isso geralmente é obtido via 'requests.get(url, headers=...).text' ou 'driver.page_source' do Selenium.")
    
    # --- SIMULAÇÃO DE HTML (SUBSTITUA PELA SUA REQUISIÇÃO REAL) ---
    # Aqui, para fins de teste, você pode colar um HTML capturado manualmente do navegador
    # ou usar um HTML real de um teste anterior com sucesso.
    # Por exemplo, se você salvou o response.text em um arquivo:
    # with open("cambridge_elegance.html", "r", encoding="utf-8") as f:
    #     html_content_for_test = f.read()
    # OU
    # Faça a requisição aqui:
    import requests
    # Use SEU User-Agent aqui!
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
        "Accept-Language": "en-US,en;q=0.9,pt-BR;q=0.8,pt;q=0.7",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
    }
    
    word_to_process = "elegance" # Ou "vision", "machine learning", etc.
    target_url = f"https://dictionary.cambridge.org/dictionary/english/{word_to_process}"

    html_content_for_test = ""
    try:
        response_test = requests.get(target_url, headers=headers, timeout=10)
        response_test.raise_for_status()
        html_content_for_test = response_test.text
        print(f"\nConteúdo HTML obtido com sucesso para '{word_to_process}'.")
    except requests.exceptions.RequestException as e:
        print(f"\nErro ao obter HTML para '{word_to_process}'. Por favor, verifique a requisição ou use Selenium: {e}")
        exit() # Sai se não conseguir o HTML
    
    # --- FIM DA SIMULAÇÃO ---

    extractor = CambridgeDefinitionExtractor(html_content_for_test)
    all_extracted_data = extractor.extract_all_data()

    import json
    print("\n--- Dados Extraídos ---")
    print(json.dumps(all_extracted_data, indent=2, ensure_ascii=False))

    # Exemplo de como acessar alguns dados
    print(f"\nPalavra Principal: {all_extracted_data.get('word_title')}")
    print("\nPronúncias:")
    for pron in all_extracted_data.get('pronunciations', []):
        print(f"  - Região: {pron.get('region')}, IPA: {pron.get('ipa')}, Áudio: {pron.get('audio_url')}")

    print("\nSignificados:")
    for meaning_block in all_extracted_data.get('meanings', []):
        print(f"  Parte da Fala: {meaning_block.get('part_of_speech')}")
        if meaning_block.get('guideword'):
            print(f"    Guia de Sentido: {meaning_block.get('guideword')}")
        for definition in meaning_block.get('definitions', []):
            print(f"    - Definição: {definition.get('text')}")
            if definition.get('synonyms'):
                print(f"      Sinônimos: {', '.join(definition.get('synonyms'))}")
            if definition.get('antonyms'):
                print(f"      Antônimos: {', '.join(definition.get('antonyms'))}")
            for example in definition.get('examples', []):
                print(f"      Exemplo: \"{example}\"")

    print("\nPhrasal Verbs e Expressões Idiomáticas:")
    for expr_block in all_extracted_data.get('phrasal_verbs_and_idioms', []):
        print(f"  - Frase: {expr_block.get('phrase')}")
        for definition in expr_block.get('definitions', []):
            print(f"    Definição: {definition.get('text')}")
            for example in definition.get('examples', []):
                print(f"      Exemplo: \"{example}\"")

    print("\nExpressões Relacionadas:")
    for related_block in all_extracted_data.get('related_expressions', []):
        print(f"  Seção: {related_block.get('section_title')}")
        for item in related_block.get('items', []):
            print(f"    - {item}")

Por favor, certifique-se de que o 'html_content' é o HTML completo da página.
Isso geralmente é obtido via 'requests.get(url, headers=...).text' ou 'driver.page_source' do Selenium.

Conteúdo HTML obtido com sucesso para 'elegance'.

--- Dados Extraídos ---
{
  "word_title": null,
  "pronunciations": [
    {
      "region": "uk",
      "ipa": "ˈel.ə.ɡəns",
      "audio_url": "/media/english/uk_pron/u/uke/ukele/ukelect007.mp3"
    },
    {
      "region": "us",
      "ipa": "ˈel.ə.ɡəns",
      "audio_url": "/media/english/us_pron/e/ele/elega/elegance.mp3"
    }
  ],
  "meanings": [],
  "phrasal_verbs_and_idioms": [],
  "related_expressions": []
}

Palavra Principal: None

Pronúncias:
  - Região: uk, IPA: ˈel.ə.ɡəns, Áudio: /media/english/uk_pron/u/uke/ukele/ukelect007.mp3
  - Região: us, IPA: ˈel.ə.ɡəns, Áudio: /media/english/us_pron/e/ele/elega/elegance.mp3

Significados:

Phrasal Verbs e Expressões Idiomáticas:

Expressões Relacionadas:


In [12]:
{"word": word, "definitions": definitions}

{'word': 'elegance',
 'definitions': [{'definition': 'the quality of being graceful and attractive in appearance or behaviour:',
   'examples': ['It was her natural elegance that struck me.',
    'the elegance of her clothes']}]}